In [1]:
import pandas as pd
import numpy as np

In [2]:
log = pd.read_csv("../raw_data/general_check_up.csv")

In [3]:
log.head()

,patient_id,activity_name,start_timestamp,end_timestamp,result_ready_time,gender,marital_status
0,0,Registration,0.00,5.29,NaN,Male,NaN
1,1,Registration,1.18,6.66,NaN,Male,NaN
2,2,Registration,5.29,10.81,NaN,Female,Single
3,0,Payment,5.29,7.97,NaN,Male,NaN
4,3,Registration,6.66,11.67,NaN,Male,NaN


In [6]:
# Bước 1: Sắp xếp
log = log.sort_values(by=['patient_id', 'start_timestamp']).reset_index(drop=True)

# Bước 2: Tính Thời gian Chờ
def calculate_wait_time(group):
    # Thời điểm kết thúc của hoạt động trước đó (dịch chuyển 1 hàng)
    prev_end_time = group['end_timestamp'].shift(1)
    
    # Tính thời gian chờ: Start Time hiện tại - End Time trước đó
    # Đây là thời gian chờ BẮT ĐẦU hoạt động hiện tại
    wait_time = group['start_timestamp'] - prev_end_time
    
    # Xử lý hàng đầu tiên của mỗi case: Wait Time = 0
    wait_time.iloc[0] = 0.0
    
    # Lưu ý: Các giá trị NaN có thể xuất hiện nếu có lỗi dữ liệu, nên dùng fillna(0)
    # Tuy nhiên, chỉ có hàng đầu tiên là NaN theo logic shift, nên ta dùng iloc[0]=0
    
    return wait_time

# Áp dụng hàm tính toán cho từng Patient ID
log['wait_time'] = log.groupby('patient_id').apply(calculate_wait_time).reset_index(level=0, drop=True)

print(log)

      patient_id                 activity_name  start_timestamp  \
0              0                  Registration             0.00   
1              0                       Payment             5.29   
2              0             Get Triage Number             7.97   
3              0           Measure Vital Signs             8.87   
4              0  General Medicine Examination            13.87   
...          ...                           ...              ...   
1965          99                    Urine Test          2557.45   
1966          99            Cardiac Ultrasound          2890.19   
1967          99            General Ultrasound          4683.40   
1968          99                    Blood Test          4730.17   
1969          99                    Conclusion          4829.07   

      end_timestamp  result_ready_time  gender marital_status  wait_time  
0              5.29                NaN    Male            NaN       0.00  
1              7.97                NaN    Mal

/tmp/ipykernel_9131/609727918.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  log['wait_time'] = log.groupby('patient_id').apply(calculate_wait_time).reset_index(level=0, drop=True)


In [7]:
wait_time_min = log['wait_time'].min()
wait_time_max = log['wait_time'].max()

In [8]:
log['reward'] = round(1 - (log['wait_time'] - wait_time_min) / (wait_time_max - wait_time_min), 2)

In [9]:
def generate_prefix(row, log):
    id = row['patient_id']
    activity = row['activity_name']
    log_of_case = log[log['patient_id'] == id]
    activities = log_of_case['activity_name']
    prefix = []
    for act in activities:
        prefix.append(act)
        if act == activity:
            break
    return prefix

In [10]:
log['prefix'] = log.apply(lambda row: generate_prefix(row, log), axis=1)

In [11]:
log['marital_status'] = log['marital_status'].fillna('None')

In [12]:
log['state'] = log.apply(lambda row: [row['gender'], row['marital_status'], row['prefix']], axis=1)


In [13]:
data = []

for pid, group in log.groupby('patient_id'):
    group = group.sort_values('start_timestamp').reset_index(drop=True)
    activities = group['activity_name'].tolist()
    states = group['state'].tolist()
    rewards = group['reward'].tolist()
    
    # Lấy thông tin tĩnh từ state đầu tiên
    gender = states[0][0]
    marital_status = states[0][1]
    
    # --- Bước khởi đầu (initial transition) ---
    initial_state = [gender, marital_status, []]
    first_action = activities[0]
    first_next_state = states[0]
    first_reward = rewards[0]
    
    data.append({
        'patient_id': pid,
        'state': initial_state,
        'action': first_action,
        'next_state': first_next_state,
        'reward': first_reward
    })
    
    # --- Các bước i -> i+1 ---
    for i in range(len(group) - 1):
        state = states[i]
        action = activities[i + 1]
        next_state = states[i + 1]
        reward = rewards[i + 1]
        
        data.append({
            'patient_id': pid,
            'state': state,
            'action': action,
            'next_state': next_state,
            'reward': reward
        })

# Tạo DataFrame RL
rl_data = pd.DataFrame(data)

In [15]:
rl_data.to_csv("general_check_up_rl_data.csv", index=False)

In [16]:
import d3rlpy

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g

In [ ]:
terminals = []

for pid, group in rl_data.groupby('patient_id'):
    n = len(group)
    terminals += [False] * (n - 1) + [True]  # chỉ dòng cuối là True

rl_data['terminal'] = terminals

In [19]:
from d3rlpy.dataset import MDPDataset
from d3rlpy.algos import CQL

dataset = MDPDataset(
    observations=rl_data['state'],
    actions=rl_data['action'],
    rewards=rl_data['reward'],
    terminals=rl_data['terminal']  # hoặc dùng giá trị thật nếu có
)

# Conservative Q-Learning (CQL) – phổ biến nhất cho offline RL
cql = CQL(use_gpu=True)
cql.fit(dataset, n_steps=100000)


ValueError: invalid observation type: <class 'pandas.core.series.Series'>

In [20]:
import gym

env = gym.make("CartPole-v1")

In [21]:
import d3rlpy

# setup algorithm
random_policy = d3rlpy.algos.DiscreteRandomPolicyConfig().create()

# prepare experience replay buffer
buffer = d3rlpy.dataset.create_fifo_replay_buffer(limit=100000, env=env)

# start data collection
random_policy.collect(env, buffer, n_steps=100000)

# save ReplayBuffer
with open("random_policy_dataset.h5", "w+b") as f:
    buffer.dump(f)

2025-10-30 09:31.09 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[()]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[[1]])
2025-10-30 09:31.09 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-10-30 09:31.09 [info     ] Action size has been automatically determined. action_size=2
2025-10-30 09:31.09 [debug    ] Building model...             
2025-10-30 09:31.09 [debug    ] Model has been built.         


  0%|          | 0/100000 [00:00<?, ?it/s]


AttributeError: module 'numpy' has no attribute 'bool8'

## Các bước thực hiện: Single Agent - Single Objective (1 tiêu chí: Tổng thời gian xử lí của 1 case):
1. Test trước với dataset bệnh viện:
- Thời gian xử lí tại 1 node = Thời gian thực thi + Thời gian chờ.
- Tổng thời gian xử lí của 1 case = Tổng Thời gian xử lí tại tất cả các node của case đó.
- Tổng thời gian xử lí của 1 case tối ưu = Tổng Thời gian thực thi của các node (nghĩa là không có node nào phải chờ.)
- Tạo dataset training offline.
- State:
    - Prefix.
    - Environment Information at time.

- Train model.
- Giả lập môi trường với simpy, điều phối bằng model đã train. -> Event logs.
- Kiểm tra event log với event logs gốc (hoặc qui trình gốc nếu không có real event logs): 
    - Kiểm tra tuân thủ qui trình
    - Kiểm tra kết quả: tổng thời gian xử lí của 1 case.

2. Test với dataset thực tế: BPIC 2019.
- Data xử lí đều dạng xes để khỏi phải chỉnh định dạng datetime. Và tên gọi các cột chính cũng cố định cho dễ sử dụng.
- Tính thời gian chờ tại mỗi node.
- Convert data về dạng offline train data.
- Train với model Q Learning.
- Cần có 1 model để tìm ra thời gian xử lí trung bình của từng node là bao nhiêu, -> có end:timestamp. -> Mới train model được.
2. Mở rộng ra với nhiều dataset khác.
-